In [4]:

from langchain import hub
a = hub.pull("rlm/rag-prompt")
print(a)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


네, AI 회의록 검색 및 요약 챗봇 프로젝트에 LangGraph의 Multi-Agent RAG 아키텍처를 적용하는 것은 매우 효과적인 접근 방식입니다. 복잡한 사용자 질문을 체계적으로 분해하고, 각 단계에 특화된 에이전트들이 협력하여 정확도 높은 답변을 생성할 수 있습니다.

아래에 Multi-Agent RAG를 활용한 구체적인 방안을 단계별로 제안해 드립니다.

### **프로젝트 목표**

사용자가 자연어 질문을 통해 특정 회의의 내용(예: 결정 사항, 특정 인물의 발언, 안건 진행 상황 등)을 정확하게 검색하고, 요약된 형태로 답변을 받는 챗봇 시스템을 구축한다.

-----

### **전체 아키텍처 구상**

전체 시스템은 **"데이터 전처리 및 인덱싱"** 단계와 **"Multi-Agent 질의응답"** 단계로 구성됩니다.

  *(아키텍처 예시 이미지)*

**1. 데이터 전처리 및 인덱싱 (RAG의 'Retrieval' 기반 마련)**

챗봇이 회의록 내용을 검색하기 위해 원본 데이터를 가공하여 Vector DB에 저장하는 단계입니다.

  * **회의록 분할 (Chunking):**

      * 하나의 긴 회의록 텍스트를 의미 있는 단위로 분할합니다.
      * **단순 분할:** 고정된 글자 수나 문장 수로 자릅니다.
      * **의미 기반 분할 (권장):** 발언자, 발언 시간, 논의된 안건을 기준으로 분할합니다. 이렇게 하면 검색 시 특정인의 발언이나 특정 주제에 대한 내용을 더 정확하게 가져올 수 있습니다.

  * **메타데이터 추출 및 저장:**

      * 각 분할된 텍스트(Chunk)에 대한 메타데이터를 추출하여 함께 저장합니다. 이는 검색 효율을 극대화합니다.
      * **필수 메타데이터:** 회의 날짜, 회의 제목, 참석자, 발언자, 안건(Agenda) 등
      * **선택적 메타데이터:** 결정 사항(Decision), 실행 항목(Action Item) 태그

  * **임베딩 및 Vector DB 저장:**

      * 분할된 각 Chunk를 임베딩 모델(e.g., `ko-sroberta-multitask`, `OpenAI text-embedding-3-small`)을 사용하여 벡터로 변환합니다.
      * 변환된 벡터를 텍스트 원본 및 메타데이터와 함께 FAISS, ChromaDB, Pinecone 같은 Vector DB에 저장(Indexing)합니다.

**2. LangGraph를 이용한 Multi-Agent 질의응답 시스템 설계**

사용자의 질문을 처리하기 위해 여러 전문 에이전트가 LangGraph로 정의된 순서와 조건에 따라 협력적으로 작동합니다.

#### **주요 에이전트(Agent) 정의**

**에이전트 1: 질문 분석 및 라우팅 에이전트 (Query Router Agent)**

  * **역할:** 사용자의 질문을 가장 먼저 받아 의도를 파악하고, 어떤 처리 경로가 적합할지 결정합니다.
  * **주요 기능:**
      * **질문 유형 분류:**
          * **단순 정보 검색:** "5월 10일 회의 참석자 알려줘."
          * **요약 요청:** "지난 주 마케팅팀 주간 회의 요약해줘."
          * **비교/분석 질문:** "A안건에 대한 김대리와 박과장의 의견 차이는 뭐야?"
      * **경로 결정 (Routing):** 분류된 의도에 따라 다음 작업(에이전트)을 호출할지 결정합니다. 예를 들어, 단순 정보 검색은 바로 '검색 에이전트'로, 복잡한 분석은 '검색 계획 에이전트'로 전달합니다.

**에이전트 2: 검색 계획 에이전트 (Search Planner Agent)**

  * **역할:** 복잡한 질문에 대해 답변을 찾기 위한 최적의 검색 전략을 수립합니다.
  * **주요 기능:**
      * **질문 분해:** "A안건에 대한 김대리와 박과장의 의견 차이"라는 질문을 아래와 같은 하위 검색어로 분해합니다.
        1.  "A안건"에 대한 회의록 검색
        2.  해당 회의록에서 "김대리"의 발언 검색
        3.  해당 회의록에서 "박과장"의 발언 검색
      * **검색 쿼리 생성:** 분해된 각 하위 질문에 대해 Vector DB에서 사용할 검색어(키워드, 벡터 검색용 문장)와 메타데이터 필터(회의 날짜, 참석자)를 조합하여 구체적인 쿼리를 생성합니다.

**에이전트 3: 검색 실행 에이전트 (Retrieval Agent with Tools)**

  * **역할:** '검색 계획 에이전트'가 수립한 전략에 따라 실제 Vector DB에서 관련 문서를 검색합니다.
  * **보유 도구 (Tools):**
      * `vector_search_tool`: 의미적으로 유사한 회의록 Chunk를 검색합니다.
      * `metadata_filter_tool`: 날짜, 참석자 등 메타데이터를 기준으로 회의록을 필터링합니다.
      * `keyword_search_tool`: 특정 키워드(고유명사 등)가 포함된 문서를 검색합니다.

**에이전트 4: 답변 생성 및 종합 에이전트 (Synthesizer & Generation Agent)**

  * **역할:** 검색된 여러 회의록 조각(Chunks)들을 종합하여 사용자의 원래 질문에 맞는 최종 답변을 생성합니다.
  * **주요 기능:**
      * **정보 종합:** 검색된 여러 정보를 취합하고, 질문의 맥락에 맞게 재구성합니다.
      * **자연어 답변 생성:** "김대리는 A안건에 대해 비용 효율성을 근거로 긍정적인 입장이었지만, 박과장은 초기 구현 리스크가 크다는 점에서 반대 의견을 제시했습니다." 와 같이 자연스러운 문장으로 답변을 생성합니다.
      * **출처 제시 (Citation):** 답변의 근거가 된 회의록의 날짜나 발언 시간 등을 함께 제시하여 신뢰도를 높입니다. (예: "이는 2025년 9월 26일 회의록 32:15 발언을 근거로 합니다.")

#### **LangGraph 구현 흐름**

1.  **상태 정의 (Define State):** 에이전트들이 공유할 데이터 구조(State)를 정의합니다.

      * `question`: 사용자의 원본 질문
      * `plan`: 검색 계획
      * `documents`: 검색된 문서 조각들
      * `answer`: 최종 생성된 답변

2.  **노드 정의 (Define Nodes):** 위에서 정의한 각 에이전트를 LangGraph의 노드(Node)로 구현합니다.

3.  **엣지 연결 (Connect Edges):** 노드 간의 작업 흐름을 엣지(Edge)로 정의합니다.

      * `START` -\> `Query Router Agent`
      * `Query Router Agent`는 질문 유형에 따라 `Search Planner Agent`로 가거나, `Retrieval Agent`로 바로 가는 **조건부 엣지(Conditional Edge)** 를 가집니다.
      * `Search Planner Agent` -\> `Retrieval Agent`
      * `Retrieval Agent` -\> `Synthesizer & Generation Agent`
      * `Synthesizer & Generation Agent` -\> `END`

-----

### **기대 효과**

  * **정확성 향상:** 복잡한 질문을 분해하고 체계적으로 검색하여 단일 RAG 방식보다 훨씬 정확하고 깊이 있는 답변 생성이 가능합니다.
  * **모듈성 및 확장성:** 각 에이전트의 역할을 명확히 분리하여 특정 기능(예: 검색 성능)만 개선하거나 새로운 기능(예: Action Item을 바로 Jira 티켓으로 생성하는 에이전트)을 추가하기 용이합니다.
  * **디버깅 용이:** LangGraph의 시각화 도구(LangSmith 등)를 활용하면, 어떤 에이전트가 어떤 결정을 내리고 어떤 결과를 반환했는지 단계별로 추적할 수 있어 문제 해결이 쉽습니다.

이러한 방안을 기반으로 프로젝트를 진행하시면, 단순 키워드 검색을 넘어 회의록 데이터의 가치를 극대화하는 고도화된 AI 회의록 챗봇을 성공적으로 구축하실 수 있을 것입니다.